# Fine-tune BLIP for Blind Reader Assistant

Trains `Salesforce/blip-image-captioning-base` on **VizWiz-Captions**, a set of photos taken by blind people with human-written descriptions, and publishes the result to your Hugging Face account for free.

1. **Runtime → Change runtime type → T4 GPU** (free).
2. Run the cells top to bottom. Training takes about 30–45 minutes.
3. Put the printed model name into `config.py` (`BLIP_MODEL`) in the app.

In [ ]:
!pip -q install -U "transformers<5" datasets accelerate
!nvidia-smi -L

Sign in with a Hugging Face **write** token from https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import notebook_login, whoami
notebook_login()

In [ ]:
HUB_REPO = f"{whoami()['name']}/blip-vizwiz"  # change the name if you like
print("Will publish to:", HUB_REPO)
!git clone -q https://github.com/AdityaWagh06/Blind_reader_app.git

Live training charts (loss going down = the model is learning). Run this before training; use the refresh icon in the panel to update.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir blip-vizwiz/logs

In [ ]:
!python Blind_reader_app/training/finetune_blip.py --hub_repo {HUB_REPO} --epochs 3 --batch_size 16

## Try the published model

In [ ]:
import requests, torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained(HUB_REPO)
model = BlipForConditionalGeneration.from_pretrained(HUB_REPO).to("cuda" if torch.cuda.is_available() else "cpu")
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/coco_sample.png"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
out = model.generate(**processor(images=image, return_tensors="pt").to(model.device), max_length=40, num_beams=3)
print(processor.decode(out[0], skip_special_tokens=True))
image